In [1]:
%reload_ext autoreload
%autoreload 2

from fpp.models.np_model_cmp import NPModelCMP

/n/home07/yitians/.conda/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
m = NPModelCMP()

In [ ]:
mu = jnp.zeros_like(m.data)

# np templates
npt_compressed = jnp.array([m.nfw_1p2])
theta = []
for ips, ps in enumerate(["nfw"]):
    A   = numpyro.sample(f"A_{ps}",   dist.Uniform(1e-4, 1.))
    n1  = numpyro.sample(f"n1_{ps}",  dist.Uniform(4., 6.))
    n2  = numpyro.sample(f"n2_{ps}",  dist.Uniform(-6., -4.))
    sb  = numpyro.sample(f"sb_{ps}", dist.Uniform(5., 20.))
    theta.append([A, n1, n2, sb])
theta = jnp.array(theta)


# likelihoods and exposure
# Pad the last exposure region so that all are the same size
exp_lens = [len(self.expreg_indices[i]) for i in range(len(self.expreg_indices))]
n_pad = exp_lens[0] - exp_lens[-1]

expreg_indices = jnp.zeros_like(self.expreg_indices)
expreg_indices = expreg_indices.at[:-1].set(self.expreg_indices[:-1])
expreg_indices = expreg_indices.at[-1].set(jnp.pad(self.expreg_indices[-1], (0, n_pad)))

log_like_np_exp_vmapped = jax.vmap(log_like_np, in_axes=(0, 0, 1, 0, None, None, None, None))
        
# Get relevant arrays for different exposure regions
mu_batch = mu[~self.mask_roi][jnp.array(expreg_indices)]
npt_compressed_batch = npt_compressed[:, ~self.mask_roi][:, jnp.array(expreg_indices)]
data_batch = self.data[~self.mask_roi][jnp.array(expreg_indices)]
exposure_multiplier = self.exposure_means_list / self.exposure_mean

theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
# theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])

with numpyro.plate("data", size=len(mu[~self.mask_roi]), dim=-1):
    
    log_like_exp = log_like_np_exp_vmapped(theta, mu_batch, npt_compressed_batch, data_batch, self.f_ary, self.df_rho_ary, self.k_max, len(expreg_indices[0]))
    loglike = jnp.concatenate(log_like_exp)[:len(mu[~self.mask_roi])]

    with handlers.mask(mask=~jnp.logical_or(jnp.isinf(loglike), jnp.isnan(loglike))):
        return numpyro.factor('log-likelihood', loglike)